In [ ]:
from typing import Optional
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from core.ai import get_llm
from core.db import db
from enum import Enum

# from langchain_core.globals import set_debug
# set_debug(False)
llm = get_llm()

c:\Users\Pachara Auikim\Desktop\Graph_RAG\graph_rag_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


...Loading LLM model
Loaded LLM model.


In [2]:
def display_usage(usage):
    print("\n--- Token Usage ---")
    print(f"Input Tokens  : {usage['input_tokens']}")
    print(f"Output Tokens : {usage['output_tokens']}")
    print(f"Total Tokens  : {usage['total_tokens']}")
    
def show_index():
 with db.get_session() as session:
    a = session.run("show indexes")
    for i in a.data(): 
      print(f"type:{i['type']} name:{i['name']}")

In [ ]:
class QuestionCategory(str, Enum):
    PERSONAL_CONTACT = "personal_contact"    
    MOU_STATUS = "mou_status"
    COMPANY_RELATIONSHIP = "company_relationship"
    COMPANY_ACTIVITY = "company_activity"
    OTHER = "other"

class QuestionClassifier(BaseModel):
    category: QuestionCategory = Field(description="""หมวดหมู่ของคำถาม""")
    person_name: Optional[str] = Field(description="ชื่อบุคคลที่ถูกกล่าวถึงในคำถาม (ถ้ามี)")
    company_name: Optional[str] = Field(description="ชื่อบริษัทหรือองค์กรที่ถูกกล่าวถึง (ถ้ามี)")
    
    
QUESTION_CLASSIFY_PROMPT = """
คุณคือ AI ผู้เชี่ยวชาญด้านการสกัดข้อมูลและจัดหมวดหมู่คำถาม
        
[ภารกิจ]
วิเคราะห์คำถามของผู้ใช้ และสกัดข้อมูลลงในโครงสร้างที่กำหนด
- วิเคราะห์ข้อมูลเฉพาะที่มีอยู่จริงในข้อความเท่านั้น ห้ามคาดเดาหรือสร้างข้อมูลขึ้นมาเอง

[กฎการจัดหมวดหมู่คำถาม]
- personal_contact: คำถามที่ต้องการขอข้อมูลติดต่อบุคคล เช่น อีเมล เบอร์โทรศัพท์
- mou_status: คำถามเกี่ยวกับสถานะ สัญญา ข้อตกลง หรือเอกสาร MOU ของบริษัท
- company_relationship: คำถามเกี่ยวกับความสัมพันธ์ เครือข่ายการติดต่อ หรือผู้เชื่อมโยงระหว่างบริษัทกับบุคคล เช่น 'บริษัท A มาจากทางไหน', 'บริษัท A มาจาก/สนิทกับอาจารย์ท่านไหน'
- company_activity: คำถามเกี่ยวกับประวัติการเข้าร่วมกิจกรรม หรือความร่วมมือต่างๆ ของบริษัท
- other: คำถามทั่วไปที่ไม่เข้าพวก
"""

prompt_template = ChatPromptTemplate.from_messages([
    ("system", QUESTION_CLASSIFY_PROMPT),
    ("user", "Text:\n{text}")
])

structured_llm = llm.with_structured_output(QuestionClassifier, include_raw=True)
question_classify_chain = prompt_template | structured_llm

question = "ขอเบอร์คุณพชร"
response = question_classify_chain.invoke({"text": question})

result = response["parsed"].model_dump()
usage = response["raw"].usage_metadata

print(result)
display_usage(usage)

In [ ]:
show_index()

In [ ]:
# with db.get_session() as session:
#     session.run("""
# match (n:Coordinator)
# detach delete n
# """)
#     # aliases

In [ ]:
with db.get_session() as session:
    session.run("""
CREATE FULLTEXT INDEX companyNameIndex IF NOT EXISTS
FOR (n:Company)
ON EACH [n.nameEn, n.nameTh, n.aliases]
""")

## Search Person

In [1]:
from search import search_person, person_format
result = search_person("golf")
print(person_format(result))

### Context

**บริษัท:** Kasikornbank - ธนาคารกสิกรไทย
- (HR) อนันต์ แซ่ตั้ง (Anan Saetang) ชื่อเล่น: Golf

**บริษัท:** Advanced Info Service - บริษัท แอดวานซ์ อินโฟร์ เซอร์วิส จำกัด (มหาชน)
- (HR) กิตติศักดิ์ รัตนพันธ์ (Kittisak Rattanapan) ชื่อเล่น: Golf


## Search Company

In [23]:
from search import search_company
    
result = search_company("scbx")
for i in result:
    print(i)

{'nameTh': 'บริษัท เอสซีบี เอกซ์ จำกัด (มหาชน)', 'nameEn': 'SCB X Public Company Limited', 'aliases': ['SCB', 'SCBX', 'ไทยพาณิชย์'], 'score': 5.281850337982178}
{'nameTh': 'เครือซิเมนต์ไทย', 'nameEn': 'Siam Cement Group', 'aliases': ['SCG', 'เอสซีจี', 'ปูนใหญ่'], 'score': 0.33741891384124756}


## Test retrieve context

In [ ]:
from ai import ai_assistant, get_prompt

question = "ขอเบอร์ติดต่อคุณ Golf"
context = """
### Context

**บริษัท:** Kasikornbank - ธนาคารกสิกรไทย
- (HR) อนันต์ แซ่ตั้ง (Anan Saetang) ชื่อเล่น: Golf เบอร์โทร: 065-959-0672

**บริษัท:** Advanced Info Service - บริษัท แอดวานซ์ อินโฟร์ เซอร์วิส จำกัด (มหาชน)
- (HR) กิตติศักดิ์ รัตนพันธ์ (Kittisak Rattanapan) ชื่อเล่น: Golf เบอร์โทร: 066-595-5599
"""
    
prompt = get_prompt(context, question)
response = llm.invoke(prompt)
print(response.content[0]['text'])

## Test Query

In [3]:
from cypher_query import CREATE_COMPANIES
with db.get_session() as session:
    result = session.run(CREATE_COMPANIES)

In [ ]:
creat_data = """
match (n:HR)
order by n.nameTh
return n.nameTh as nameTh, n.nickname as nickname
"""

with db.get_session() as session:
    result = session.run(creat_data)
    for i in result.data():
        print(f"ชื่อ: {i['nameTh']}, ชื่อเล่น: {i['nickname']}")

In [ ]:
creat_data = """
match (n:Company)
order by n.nameTh
return n.nameTh as nameTh, n.alias1 as alias1, n.alias2 as alias2, n.alias3 as alias3
"""

with db.get_session() as session:
    result = session.run(creat_data)
    for i in result.data():
        print(i)